In [17]:
import pandas as pd
import os
import json

dataset_id = "HuggingFaceFW/fineweb"
dataset_config = "sample-10BT"
configs_file = "config.json"
compression_methods = ["snappy", "gzip", "brotli", "lz4", "zstd"]
config_dict = json.load(open(configs_file, "r"))[dataset_id]

In [18]:
print(config_dict)

{'configs': [{'file': 'sample-10BT', 'status': 'downloaded', 'download_datetime': '2025-05-02 14:40:06', 'path': './datasets/HuggingFaceFW/fineweb/sample/10BT', 'size_original': 30639384917, 'size_snappy': 2036861286, 'size_gzip': 1304555928, 'size_brotli': 1117233751, 'size_lz4': 2078916387, 'size_zstd': 1427266717, 'rows': 1000000, 'compress': [{'function': "dump=file_path.split('/')[4].split('/')[0]", 'size_compression_snappy': 2036850933, 'size_compression_gzip': 1304549930, 'size_compression_brotli': 1117228637, 'size_compression_lz4': 2078905984, 'size_compression_zstd': 1427259905}, {'function': 'file_path=file_path+dump+file_path', 'size_compression_snappy': 2035496849, 'size_compression_gzip': 1304414401, 'size_compression_brotli': 1117108748, 'size_compression_lz4': 2078761433, 'size_compression_zstd': 1427074982}, {'function': 'combine all string columns', 'size_compression_snappy': 2064808011, 'size_compression_gzip': 1337756388, 'size_compression_brotli': 1142616262, 'size

In [19]:
def get_df(path, max_rows=1_000_000):
    parts = []
    total_rows = 0

    for fname in sorted(os.listdir(path)):
        df_part = pd.read_parquet(os.path.join(path, fname))
        n = len(df_part)

        if total_rows + n >= max_rows:
            # Only take what we still need to reach the cap
            df_part = df_part.iloc[: max_rows - total_rows]
            parts.append(df_part)
            break

        parts.append(df_part)
        total_rows += n

        if total_rows >= max_rows:
            break

    return pd.concat(parts, ignore_index=True)

In [20]:
df = get_df(config_dict['configs'][0]['path'])
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   text            1000000 non-null  object 
 1   id              1000000 non-null  object 
 2   dump            1000000 non-null  object 
 3   url             1000000 non-null  object 
 4   date            1000000 non-null  object 
 5   file_path       1000000 non-null  object 
 6   language        1000000 non-null  object 
 7   language_score  1000000 non-null  float64
 8   token_count     1000000 non-null  int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 68.7+ MB
None


In [21]:
os.makedirs("datasets_compress/HuggingFaceFW/fineweb/", exist_ok=True)

In [22]:
def compare_dataframes(df1, df2):
    # Align columns by name
    df1_sorted = df1.sort_index(axis=1)
    df2_sorted = df2[df1_sorted.columns]  # reorder df2 to match df1

    # Check shape first
    if df1_sorted.shape != df2_sorted.shape:
        print("DataFrames have different shapes:", df1_sorted.shape, df2_sorted.shape)
        return

    # Create a boolean DataFrame of element-wise comparisons
    diff = df1_sorted != df2_sorted

    if not diff.any().any():
        print("DataFrames are equal")
    else:
        print("Differences found at these locations:")
        differing_cells = diff.stack()[diff.stack()]  # True where differences exist
        for (idx, col) in differing_cells.index:
            print(f"- Row {idx}, Column '{col}': df1 = {df1_sorted.at[idx, col]!r}, df2 = {df2_sorted.at[idx, col]!r}")

def get_size(start_path = '.'):
    # check is it a file or directory
    if os.path.isfile(start_path):
        return os.path.getsize(start_path)
    total = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

In [64]:

from sklearn.linear_model import LinearRegression
import numpy as np

alpha = 0.001

compress_df = df.copy()

# Compute text length
compress_df["text_len"] = compress_df["text"].str.len()

# Fit linear model
X = compress_df[["text_len"]]
y = compress_df["token_count"]
model = LinearRegression()
model.fit(X, y)

slope = model.coef_[0]
intercept = model.intercept_
print(f"Model: y = {intercept:.2f} + {slope:.2f} * x")
compress_df["predicted_token_count"] = model.predict(X).round().astype(int)
full_offset = compress_df["token_count"] - compress_df["predicted_token_count"]
token_range = compress_df["token_count"].max() - compress_df["token_count"].min()
threshold = alpha * token_range

compress_df["token_count_offset"] = full_offset.where(full_offset.abs() <= threshold, np.nan)
compress_df["offset_outlier"] = full_offset.where(full_offset.abs() > threshold, np.nan)

coverage = compress_df["offset_outlier"].isna().mean()
print(f"Offset coverage (within ±{alpha:.2%} range): {coverage:.2%}")

print(f"Max token count offset: {compress_df['token_count_offset'].max()}")
print(f"Min token count offset: {compress_df['token_count_offset'].min()}")
print(f"Mean token count offset: {compress_df['token_count_offset'].mean()}")
# index of the max token count offset
max_offset_index = compress_df["token_count_offset"].idxmax()
print(f"Max token count offset index: {max_offset_index}")
print(f"Max token count offset: {compress_df['token_count_offset'][max_offset_index]}")

# num_non_outliers = (~compress_df["is_outlier"]).sum()
# total = len(compress_df)
# coverage = num_non_outliers / total
# print(f"Number of non-outliers: {num_non_outliers}")
# print(f"Total number of rows: {total}")

# print(f"Offset coverage (within threshold): {coverage:.2%}")

compress_df = compress_df.drop(columns=["text_len", "predicted_token_count", "token_count"])
# print(compress_df.head())

Model: y = -7.52 + 0.23 * x
Offset coverage (within ±0.10% range): 91.78%
Max token count offset: 130.0
Min token count offset: -130.0
Mean token count offset: 0.8548319787833785
Max token count offset index: 2026
Max token count offset: 130.0


In [65]:
import matplotlib.pyplot as plt

plt.scatter(compress_df["text_len"], compress_df["token_count"], alpha=0.5)
plt.plot(compress_df["text_len"], model.predict(X), color='red', linewidth=2)
plt.xlabel("Text Length")
plt.ylabel("Token Count")
plt.title("Token Count vs Text Length")
plt.show()

KeyError: 'text_len'

In [ ]:
print(compress_df.head())

                                                text  \
0  |Viewing Single Post From: Spoilers for the We...   
1  *sigh* Fundamentalist community, let me pass o...   
2  A novel two-step immunotherapy approach has sh...   
3  Free the Cans! Working Together to Reduce Wast...   
4  ORLANDO, Fla. — While the Rapid Recall Exchang...   

                                                id             dump  \
0  <urn:uuid:39147604-bfbe-4ed5-b19c-54105f8ae8a7>  CC-MAIN-2013-20   
1  <urn:uuid:ba819eb7-e6e6-415a-87f4-0347b6a4f017>  CC-MAIN-2013-20   
2  <urn:uuid:07b8e00d-b445-4736-a593-cd1c147dce21>  CC-MAIN-2013-20   
3  <urn:uuid:c970d9a2-a5ce-4050-9ea3-58d7bbd609a8>  CC-MAIN-2013-20   
4  <urn:uuid:5c2cac9e-2fda-4194-959b-6ede0668ad2a>  CC-MAIN-2013-20   

                                                 url                  date  \
0  http://daytimeroyaltyonline.com/single/?p=8906...  2013-05-18T05:48:59Z   
1  http://endogenousretrovirus.blogspot.com/2007/...  2013-05-18T06:43:03Z   
2 

In [66]:
import pyarrow as pa
import pyarrow.parquet as pq

def write_parquet(df, path, PARQUET_COMPRESSION_TYPE="snappy"):
    table = pa.Table.from_pandas(df)
    pq.write_table(table, path, compression=PARQUET_COMPRESSION_TYPE)

# compress_df = df.copy()
# compress_df["url_prefix"] = compress_df["url"].str.split("/").str[:3].str.join("/")
# compress_df["url"] = compress_df["url"].str.split("/").str[3:].str.join("/")
# compress_df["combine"] = compress_df['url'] + compress_df['text']
# compress_df[f"url_len"] = compress_df['url'].str.len()
# compress_df = compress_df.drop(columns=["url", "text"])

write_parquet(compress_df, f"datasets_compress/{dataset_id}/{dataset_config}.parquet")

KeyboardInterrupt: 

In [ ]:
compress_df = pd.read_parquet(f"datasets_compress/{dataset_id}/{dataset_config}.parquet")
print(compress_df.info())
print(compress_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 10 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   id              1000000 non-null  object 
 1   dump            1000000 non-null  object 
 2   date            1000000 non-null  object 
 3   file_path       1000000 non-null  object 
 4   language        1000000 non-null  object 
 5   language_score  1000000 non-null  float64
 6   token_count     1000000 non-null  int64  
 7   url_prefix      1000000 non-null  object 
 8   combine         1000000 non-null  object 
 9   url_len         1000000 non-null  int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 76.3+ MB
None
                                                id             dump  \
0  <urn:uuid:39147604-bfbe-4ed5-b19c-54105f8ae8a7>  CC-MAIN-2013-20   
1  <urn:uuid:ba819eb7-e6e6-415a-87f4-0347b6a4f017>  CC-MAIN-2013-20   
2  <urn:uuid:07b8e00d-b445-4736-a593-cd1c14

In [ ]:
size = config_dict['configs'][0]['size_snappy']
compression_size = get_size(f"datasets_compress/{dataset_id}/{dataset_config}.parquet")
print(f"Size of the dataset: {size / (1024):.2f} kB")
print(f"Size of the dataset: {size} B")
print(f"Size of the compression: {compression_size / (1024):.2f} kB")
print(f"Size of the compression: {compression_size} B")
print(f"Compression ratio: {((size-compression_size) / size) * 100:.2f} %")

Size of the dataset: 1989122.35 kB
Size of the dataset: 2036861286 B
Size of the compression: 1988755.62 kB
Size of the compression: 2036485757 B
Compression ratio: 0.02 %


In [ ]:
compress_df["id"] = "<urn:uuid:" + compress_df["id_part1"] + "-" + compress_df["id_part2"] + "-" + compress_df["id_part3"] + "-" + compress_df["id_part4"] + "-" + compress_df["id_part5"] + ">"
compress_df = compress_df.drop(columns=["id_part1", "id_part2", "id_part3", "id_part4", "id_part5"])

In [76]:
print(compress_df.head())

                                                text             dump  \
0  |Viewing Single Post From: Spoilers for the We...  CC-MAIN-2013-20   
1  *sigh* Fundamentalist community, let me pass o...  CC-MAIN-2013-20   
2  A novel two-step immunotherapy approach has sh...  CC-MAIN-2013-20   
3  Free the Cans! Working Together to Reduce Wast...  CC-MAIN-2013-20   
4  ORLANDO, Fla. — While the Rapid Recall Exchang...  CC-MAIN-2013-20   

                                                 url                  date  \
0  http://daytimeroyaltyonline.com/single/?p=8906...  2013-05-18T05:48:59Z   
1  http://endogenousretrovirus.blogspot.com/2007/...  2013-05-18T06:43:03Z   
2                     http://news.cancerconnect.com/  2013-05-18T05:23:15Z   
3  http://sharingsolution.com/2009/05/23/free-the...  2013-05-18T05:49:03Z   
4  http://supermarketnews.com/food-safety/more-su...  2013-05-18T05:25:43Z   

                                           file_path language  language_score  \
0  s3://com

In [1]:
compare_dataframes(df, compress_df)

NameError: name 'compare_dataframes' is not defined

In [1]:
import pyarrow.parquet as pq
dataset_id = "HuggingFaceFW/fineweb"
dataset_config = "sample-10BT"
# Load Parquet file metadata
parquet_file = pq.ParquetFile(f"datasets_compress/{dataset_id}/{dataset_config}.parquet")

column_sizes = {}

# Loop through each row group
for row_group_idx in range(parquet_file.num_row_groups):
    row_group = parquet_file.metadata.row_group(row_group_idx)
    for col_idx in range(row_group.num_columns):
        col = row_group.column(col_idx)
        col_name = col.path_in_schema
        col_size = col.total_compressed_size  # or use total_uncompressed_size
        column_sizes[col_name] = column_sizes.get(col_name, 0) + col_size

# Display column sizes
for col_name, size in column_sizes.items():
    print(f"{col_name}: {size / 1024:.2f} KB")

text: 1870146.23 KB
id: 35983.81 KB
dump: 9.67 KB
url: 52205.48 KB
date: 8332.33 KB
file_path: 14870.00 KB
language: 0.07 KB
language_score: 5810.29 KB
token_count_offset: 1122.65 KB
offset_outlier: 245.78 KB
